In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [2]:
spark = SparkSession.builder \
    .appName("Week5_Spark_Assignment") \
    .getOrCreate()

In [3]:
df = spark.read.csv(
    "Spark_Assignment_Dataset_8200_Rows.csv",
    header=True,
    inferSchema=True
)

In [4]:
df.show(5)

+-------+----------------+-------+----------------+-----------+---------+---------+---+------------+-------------------+--------------------+--------+-------+--------+
|user_id|transaction_date| region|product_category|sale_amount|   status|     city|age|subscription|      raw_timestamp|               email|username|  price|store_id|
+-------+----------------+-------+----------------+-----------+---------+---------+---+------------+-------------------+--------------------+--------+-------+--------+
|   1654|      2025-02-27|Central|       Groceries|    3773.18|Completed|    Delhi| 21|     Premium|2025-02-27 00:47:17|user1654@example.com|user1654|3705.93|    S008|
|   1517|      2025-11-05|   West|       Furniture|    2051.78|     NULL|Hyderabad| 33|     Premium|2025-11-05 00:35:12|user1517@example.com|user1517|2251.55|    S007|
|   1781|      2025-06-22|Central|     Electronics|    4408.59|     NULL|   Mumbai| 51|       Basic|2025-06-22 03:05:24|user1781@example.com|user1781|4037.57|  

In [5]:
df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- raw_timestamp: timestamp (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: string (nullable = true)



In [6]:
print("Rows:", df.count())
print("Columns:", len(df.columns))

Rows: 8200
Columns: 14


In [7]:
print(df.columns)

['user_id', 'transaction_date', 'region', 'product_category', 'sale_amount', 'status', 'city', 'age', 'subscription', 'raw_timestamp', 'email', 'username', 'price', 'store_id']


# Q1: What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

## Answer

Traditional MapReduce writes intermediate results to disk after every processing stage, making it slow for iterative and interactive workloads.

Apache Spark overcomes these limitations through in-memory computation and optimized execution.

### Limitations of Traditional MapReduce

- Heavy disk I/O
- Slow execution
- High latency
- Complex programming model
- Multiple MapReduce jobs required for complex workflows

# Q2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

## Answer

Spark stores intermediate datasets in RAM instead of repeatedly writing them to disk.

Machine learning algorithms require multiple iterations over the same dataset. Spark caches the data in memory, allowing repeated access without disk reads.

### Benefits

- Faster execution
- Reduced disk I/O
- Efficient iterative computation
- Better resource utilization
- Improved performance for machine learning algorithms

# Q3: Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date.

In [8]:
df_no_duplicates = df.dropDuplicates(["user_id", "transaction_date"])
df_no_duplicates.show(5)

+-------+----------------+------+----------------+-----------+---------+---------+---+------------+-------------------+--------------------+--------+-------+--------+
|user_id|transaction_date|region|product_category|sale_amount|   status|     city|age|subscription|      raw_timestamp|               email|username|  price|store_id|
+-------+----------------+------+----------------+-----------+---------+---------+---+------------+-------------------+--------------------+--------+-------+--------+
|   1000|      2025-03-12| South|     Electronics|        0.0|  Pending|   Mumbai| 25|       Basic|2025-03-12 00:16:10|user1000@example.com|user1000|   NULL|    S003|
|   1000|      2025-05-06|  East|       Groceries|    3832.25|Cancelled|Hyderabad| 16|       Basic|2025-05-06 16:28:47|user1000@example.com|user1000|4555.58|    S016|
|   1000|      2025-08-01|  East|        Clothing|    2054.46|Completed|  Kolkata| 31|       Basic|2025-08-01 02:13:33|user1000@example.com|user1000|2137.52|    S018

# Q4: Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount.

In [9]:
west_sales = (
    df.filter(col("region") == "West")
      .groupBy("product_category")
      .agg(avg("sale_amount").alias("Average Sale Amount"))
)

west_sales.show()

+----------------+-------------------+
|product_category|Average Sale Amount|
+----------------+-------------------+
|       Groceries|  2409.748000000001|
|     Electronics| 2343.8249074074083|
|        Clothing| 2355.7495333333322|
| Office Supplies| 2441.3120481927726|
|       Furniture| 2364.6711940298483|
+----------------+-------------------+



# Q5: What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'.

## Answer

**.na.drop()**

- Removes rows containing null values.

**.na.fill()**

- Replaces null values with a specified value while keeping the records.


In [10]:
df_filled = df.na.fill({"status": "Unknown"})
df_filled.select("status").show(10)

+---------+
|   status|
+---------+
|Completed|
|  Unknown|
|  Unknown|
|  Unknown|
|  Unknown|
|  Unknown|
|  Unknown|
|Cancelled|
|  Pending|
|  Pending|
+---------+
only showing top 10 rows


# Q6: Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.

In [11]:
city_count = (
    df.groupBy("city")
      .count()
      .filter(col("count") > 100)
)

city_count.show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore| 1046|
|  Chennai|  993|
|   Mumbai| 1038|
|Ahmedabad| 1048|
|  Kolkata|  987|
|     Pune| 1053|
|    Delhi| 1003|
|Hyderabad| 1032|
+---------+-----+



# Q7: How does the immutability of Spark DataFrames affect how you perform "data cleaning" steps like dropping columns or renaming them?

## Answer

Spark DataFrames are immutable, meaning they cannot be modified after creation.

Whenever a transformation such as `drop()`, `withColumn()`, or `withColumnRenamed()` is performed, Spark creates a new DataFrame while leaving the original DataFrame unchanged.

### Advantages

- Prevents accidental modification
- Supports fault tolerance
- Enables lazy evaluation
- Improves optimization

# Q8: Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'.

In [12]:
premium_users = df.filter(
    (col("age").between(18, 30)) &
    (col("subscription") == "Premium")
)

premium_users.show()

+-------+----------------+-------+----------------+-----------+---------+---------+---+------------+-------------------+--------------------+--------+-------+--------+
|user_id|transaction_date| region|product_category|sale_amount|   status|     city|age|subscription|      raw_timestamp|               email|username|  price|store_id|
+-------+----------------+-------+----------------+-----------+---------+---------+---+------------+-------------------+--------------------+--------+-------+--------+
|   1654|      2025-02-27|Central|       Groceries|    3773.18|Completed|    Delhi| 21|     Premium|2025-02-27 00:47:17|user1654@example.com|user1654|3705.93|    S008|
|   1271|      2025-03-13|   East| Office Supplies|    2760.54|     NULL|   Mumbai| 19|     Premium|2025-03-13 07:47:35|user1271@example.com|user1271|2921.83|    S005|
|   1059|      2025-10-24|   East|     Electronics|    2874.65|Completed|Bangalore| 20|     Premium|2025-10-24 23:34:03|user1059@example.com|user1059|2519.22|  

# Q9: When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()?

## Answer

Handling null values before aggregation ensures accurate and reliable analytical results.

### Reasons

- Prevents inaccurate calculations
- Improves data quality
- Avoids unexpected aggregation results
- Produces reliable reports

Common approaches include using `.na.fill()` or `.na.drop()`.

# Q10: Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time.

In [13]:
df_timestamp = (
    df.withColumn(
        "raw_timestamp",
        col("raw_timestamp").cast(TimestampType())
    )
    .withColumnRenamed(
        "raw_timestamp",
        "event_time"
    )
)

df_timestamp.printSchema()
df_timestamp.show(5)

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: string (nullable = true)

+-------+----------------+-------+----------------+-----------+---------+---------+---+------------+-------------------+--------------------+--------+-------+--------+
|user_id|transaction_date| region|product_category|sale_amount|   status|     city|age|subscription|         event_time|               email|username|  price|store_id|
+-------+----------------+-------+----------------+-----------+---------+---------+---+---

# Q11: Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?

## Answer

A shuffle is the process of redistributing data across partitions so that records with the same key are processed together.

It is considered a wide transformation because data moves between multiple partitions during execution.

### Examples

- groupBy()
- join()
- distinct()
- repartition()
- orderBy()

### Characteristics

- Requires network communication
- Transfers data across partitions
- Increases execution time

# Q12: Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string.

In [14]:
clean_df = df.filter(
    col("email").isNotNull() &
    (trim(col("username")) != "")
)

clean_df.show(5)

+-------+----------------+-------+----------------+-----------+---------+---------+---+------------+-------------------+--------------------+--------+-------+--------+
|user_id|transaction_date| region|product_category|sale_amount|   status|     city|age|subscription|      raw_timestamp|               email|username|  price|store_id|
+-------+----------------+-------+----------------+-----------+---------+---------+---+------------+-------------------+--------------------+--------+-------+--------+
|   1654|      2025-02-27|Central|       Groceries|    3773.18|Completed|    Delhi| 21|     Premium|2025-02-27 00:47:17|user1654@example.com|user1654|3705.93|    S008|
|   1517|      2025-11-05|   West|       Furniture|    2051.78|     NULL|Hyderabad| 33|     Premium|2025-11-05 00:35:12|user1517@example.com|user1517|2251.55|    S007|
|   1781|      2025-06-22|Central|     Electronics|    4408.59|     NULL|   Mumbai| 51|       Basic|2025-06-22 03:05:24|user1781@example.com|user1781|4037.57|  

# Q13: How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column?

In [15]:
price_statistics = df.agg(
    min("price").alias("Minimum Price"),
    max("price").alias("Maximum Price"),
    avg("price").alias("Average Price")
)

price_statistics.show()

+-------------+-------------+-----------------+
|Minimum Price|Maximum Price|    Average Price|
+-------------+-------------+-----------------+
|        10.85|       4999.5|2490.368695652167|
+-------------+-------------+-----------------+



# Q14: In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats?

## Answer

Using `inferSchema=True` with inconsistent date formats may cause Spark to infer incorrect data types or treat the column as a string.

### Risks

- Incorrect schema detection
- Parsing errors
- Null values after conversion
- Inconsistent query results

# Q15: Write a final processing pipeline that:

1. Filters out duplicates.
2. Fills null prices with 0.
3. Groups by store_id to calculate total revenue.

In [16]:
final_pipeline = (
    df.dropDuplicates()
      .na.fill({"price": 0})
      .groupBy("store_id")
      .agg(sum("price").alias("Total Revenue"))
      .orderBy("store_id")
)

final_pipeline.show()

+--------+-----------------+
|store_id|    Total Revenue|
+--------+-----------------+
|    S001|636228.5699999998|
|    S002|        590995.48|
|    S003|683583.8499999995|
|    S004|        682388.94|
|    S005|602732.5600000003|
|    S006|695522.3899999997|
|    S007|629897.4700000004|
|    S008|644834.5999999997|
|    S009|662601.8399999999|
|    S010|        645000.63|
|    S011|594823.0399999997|
|    S012|610893.2400000003|
|    S013|612619.5800000001|
|    S014|616343.5100000001|
|    S015|615435.9599999995|
|    S016|631934.1799999999|
|    S017|615706.1700000002|
|    S018|        655560.12|
|    S019|         613718.6|
|    S020|        706729.03|
+--------+-----------------+
only showing top 20 rows
